# Milestone 13 — Multimodal Encoder Integration & Real Data Alignment

## **Overview**

Milestone 13 establishes the first full interaction between the **molecular encoder** and the **phenotype image encoder**.  
The goal is to project both modalities into a **shared latent space** and compute a **contrastive alignment loss**, similar to CLIP.  
This milestone verifies that the multimodal architecture is functioning correctly and that real data (molecules + phenotype images) can be aligned.

---

## **Objectives**

- Integrate the **SimpleMolEncoder** and **ImageEncoder** into a unified multimodal pipeline.  
- Project both embeddings into a shared **128‑dimensional latent space** using `MultimodalModel`.  
- Normalize embeddings to enable cosine‑like similarity.  
- Compute a **contrastive loss** over a batch of molecule–image pairs.  
- Validate correct tensor shapes and ensure stable training behaviour.

---

## **Key Components**

### **1. Molecular Encoder**
- RDKit Morgan fingerprints (`radius=2`, `nBits=2048`).  
- 2‑layer MLP → **256‑dim embedding**.

### **2. Image Encoder**
- Pretrained **ResNet18** backbone with classification head removed.  
- Linear projection → **256‑dim embedding**.

### **3. Multimodal Projection**
- Two linear layers map molecule and image embeddings → **128‑dim latent space**.  
- L2 normalization stabilizes similarity scoring.

### **4. Contrastive Alignment Loss**
- Computes similarity matrix between molecule and image embeddings.  
- Uses cross‑entropy over diagonal matches (positive pairs).  
- Requires **batch size ≥ 2**.

---

## **Workflow**

1. Load real molecule SMILES.  
2. Load phenotype image and apply ImageNet preprocessing.  
3. Encode molecule → `mol_emb` (256‑dim).  
4. Encode image → `img_emb` (256‑dim).  
5. Build a batch of molecule–image pairs.  
6. Forward pass through `MultimodalModel` → `z_mol`, `z_img` (128‑dim).  
7. Compute contrastive loss.  
8. Verify shapes:  
   - `z_mol`: `[batch, 128]`  
   - `z_img`: `[batch, 128]`

---

## **Results**

After correcting encoder shapes and ensuring proper batching:



In [34]:
# Import RDKit for molecule handling
from rdkit import Chem

# Import PIL for loading phenotype images
from PIL import Image

# Import torchvision transforms for preprocessing images
import torchvision.transforms as T

# Import PyTorch for tensor operations
import torch

# ------------------------------------------------------------
# ImageNet-style preprocessing pipeline for ResNet18
# ------------------------------------------------------------
# This transform ensures that phenotype images are resized,
# normalized, and converted into tensors in the exact format
# expected by pretrained ResNet18.
transform = T.Compose([
    T.Resize((224, 224)),          # Resize image to 224x224 pixels
    T.ToTensor(),                  # Convert PIL image to PyTorch tensor
    T.Normalize(                   # Normalize using ImageNet mean/std
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])


In [35]:
import torch
import torch.nn as nn
from rdkit.Chem import AllChem

# ------------------------------------------------------------
# SimpleMolEncoder
# ------------------------------------------------------------
# This encoder converts an RDKit Mol object into a fixed-size
# molecular embedding using Morgan fingerprints + a small MLP.
#
# Workflow:
#   RDKit Mol → 2048‑bit Morgan fingerprint → MLP → 256‑dim vector
# ------------------------------------------------------------
class SimpleMolEncoder(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()

        # Store the output embedding dimension
        self.emb_dim = emb_dim

        # A simple 2‑layer feed‑forward network:
        #   2048‑bit fingerprint → 512 → emb_dim (default 256)
        self.mlp = nn.Sequential(
            nn.Linear(2048, 512),  # First projection layer
            nn.ReLU(),             # Non‑linearity
            nn.Linear(512, emb_dim)  # Final embedding layer
        )

    def featurize(self, mol):
        """
        Convert an RDKit Mol into a 2048‑bit Morgan fingerprint.
        The fingerprint is returned as a float32 PyTorch tensor.
        """
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, radius=2, nBits=2048
        )
        arr = torch.tensor(fp.ToList(), dtype=torch.float32)
        return arr

    def forward(self, mol):
        """
        Full forward pass:
        1. Compute Morgan fingerprint
        2. Feed it through the MLP
        3. Return a 256‑dim embedding
        """
        fp = self.featurize(mol)
        return self.mlp(fp)


In [36]:
import torchvision.models as models

# ------------------------------------------------------------
# ImageEncoder
# ------------------------------------------------------------
# This encoder converts a phenotype image into a fixed-size
# embedding using a pretrained ResNet18 backbone followed by
# a linear projection layer.
#
# Workflow:
#   PIL Image → torchvision transform → ResNet18 → 512‑dim features
#   → Linear projection → 256‑dim embedding
# ------------------------------------------------------------
class ImageEncoder(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()

        # Load pretrained ResNet18 weights (ImageNet)
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Replace the final classification layer with Identity
        # so the network outputs raw 512‑dim features instead of class logits
        backbone.fc = nn.Identity()

        # Store the modified backbone
        self.backbone = backbone

        # Linear projection: 512‑dim → emb_dim (default 256)
        self.proj = nn.Linear(512, emb_dim)

    def forward(self, img):
        """
        Forward pass for image encoding.

        Parameters:
            img: A preprocessed image tensor.
                 Expected shape:
                     - [3, H, W] for a single image
                     - [B, 3, H, W] for a batch

        Steps:
            1. Add batch dimension if needed
            2. Extract features using ResNet18
            3. Project features to emb_dim (256)
        """
        # If input is a single image (no batch dimension), add one
        if img.ndim == 3:
            img = img.unsqueeze(0)

        # Extract 512‑dim features from ResNet18
        features = self.backbone(img)

        # Project features to 256‑dim embedding
        return self.proj(features)


In [37]:
import torch.nn.functional as F

# ------------------------------------------------------------
# MultimodalModel
# ------------------------------------------------------------
# This model aligns molecule embeddings and image embeddings
# into a shared latent space using:
#   - Two linear projection layers (mol_proj, img_proj)
#   - L2 normalization
#   - A CLIP-style contrastive loss
#
# Workflow:
#   mol_emb (256) → mol_proj → z_mol (128)
#   img_emb (256) → img_proj → z_img (128)
#   Normalize → similarity matrix → contrastive loss
# ------------------------------------------------------------
class MultimodalModel(nn.Module):
    def __init__(self, mol_dim=256, img_dim=256, proj_dim=128):
        super().__init__()

        # Linear projection for molecule embeddings (256 → 128)
        self.mol_proj = nn.Linear(mol_dim, proj_dim)

        # Linear projection for image embeddings (256 → 128)
        self.img_proj = nn.Linear(img_dim, proj_dim)

        # Temperature parameter for scaling similarities
        # Initialized to log(1/0.07), following CLIP convention
        self.logit_scale = nn.Parameter(
            torch.ones([]) * torch.log(torch.tensor(1/0.07))
        )

    def forward(self, mol_emb, img_emb):
        """
        Forward pass:
        1. Project molecule and image embeddings into shared space
        2. Normalize both embeddings (unit length)
        3. Return aligned embeddings
        """
        # Project embeddings into 128‑dim latent space
        z_mol = self.mol_proj(mol_emb)
        z_img = self.img_proj(img_emb)

        # Normalize embeddings for cosine-like similarity
        z_mol = F.normalize(z_mol, dim=-1)
        z_img = F.normalize(z_img, dim=-1)

        return z_mol, z_img

    def contrastive_loss(self, z_mol, z_img):
        """
        Compute CLIP-style contrastive loss:
        - Similarity matrix = z_mol @ z_img.T
        - Scale logits using learned temperature
        - Cross-entropy over diagonal matches
        - Symmetric loss: molecule→image and image→molecule
        """
        # Similarity matrix between all molecule-image pairs
        logits = z_mol @ z_img.T

        # Apply temperature scaling
        logits = logits * self.logit_scale.exp()

        # Ground-truth labels: diagonal entries are positive pairs
        labels = torch.arange(logits.size(0)).to(logits.device)

        # Cross-entropy in both directions
        loss_mol = F.cross_entropy(logits, labels)
        loss_img = F.cross_entropy(logits.T, labels)

        # Final symmetric contrastive loss
        return (loss_mol + loss_img) / 2


In [38]:
# ------------------------------------------------------------
# Instantiate all encoders and the multimodal alignment model
# ------------------------------------------------------------

# Molecular encoder:
# Converts RDKit Mol objects → Morgan fingerprints → 256‑dim embeddings
mol_encoder = SimpleMolEncoder()

# Image encoder:
# Converts phenotype images → ResNet18 features → 256‑dim embeddings
img_encoder = ImageEncoder()

# Multimodal model:
# Projects molecule + image embeddings → shared 128‑dim latent space
# and provides CLIP‑style contrastive alignment loss
model = MultimodalModel()


In [39]:
import torch.nn.functional as F

# ------------------------------------------------------------
# MultimodalModel
# ------------------------------------------------------------
# This model aligns molecule embeddings and image embeddings
# into a shared latent space using:
#   • Two linear projection layers (mol_proj, img_proj)
#   • L2 normalization for cosine‑like similarity
#   • A CLIP‑style contrastive loss for cross‑modal alignment
#
# Workflow:
#   mol_emb (256) → mol_proj → z_mol (128)
#   img_emb (256) → img_proj → z_img (128)
#   Normalize → similarity matrix → contrastive loss
# ------------------------------------------------------------
class MultimodalModel(nn.Module):
    def __init__(self, mol_dim=256, img_dim=256, proj_dim=128):
        super().__init__()

        # Linear projection for molecule embeddings (256 → 128)
        self.mol_proj = nn.Linear(mol_dim, proj_dim)

        # Linear projection for image embeddings (256 → 128)
        self.img_proj = nn.Linear(img_dim, proj_dim)

        # Temperature parameter for scaling similarities
        # Initialized following CLIP convention: log(1/0.07)
        self.logit_scale = nn.Parameter(
            torch.ones([]) * torch.log(torch.tensor(1/0.07))
        )

    def forward(self, mol_emb, img_emb):
        """
        Forward pass:
        1. Project molecule and image embeddings into shared space
        2. Normalize both embeddings to unit length
        3. Return aligned embeddings
        """
        # Project embeddings into 128‑dim latent space
        z_mol = self.mol_proj(mol_emb)
        z_img = self.img_proj(img_emb)

        # Normalize embeddings for cosine-like similarity
        z_mol = F.normalize(z_mol, dim=-1)
        z_img = F.normalize(z_img, dim=-1)

        return z_mol, z_img

    def contrastive_loss(self, z_mol, z_img):
        """
        Compute CLIP-style contrastive loss:
        - Similarity matrix = z_mol @ z_img.mT
        - Scale logits using learned temperature
        - Cross-entropy over diagonal matches (positive pairs)
        - Symmetric loss: molecule→image and image→molecule
        """
        # Similarity matrix between all molecule-image pairs
        logits = z_mol @ z_img.mT   # correct transpose for batches

        # Apply temperature scaling
        logits = logits * self.logit_scale.exp()

        # Ground-truth labels: diagonal entries are positive pairs
        labels = torch.arange(logits.size(0), device=logits.device)

        # Cross-entropy in both directions
        loss_mol = F.cross_entropy(logits, labels)
        loss_img = F.cross_entropy(logits.T, labels)

        # Final symmetric contrastive loss
        return (loss_mol + loss_img) / 2


In [40]:
# ------------------------------------------------------------
# Forward pass through the multimodal model
# ------------------------------------------------------------
# mol_batch : batch of molecule embeddings (shape [B, 256])
# img_batch : batch of image embeddings (shape [B, 256])
#
# The model projects both into a shared 128‑dimensional space
# and returns:
#   z_mol : molecule embeddings in shared space
#   z_img : image embeddings in shared space
# ------------------------------------------------------------
z_mol, z_img = model(mol_batch, img_batch)

# Print the shapes to verify correct alignment
print("z_mol:", z_mol.shape)   # Expected: [batch_size, 128]
print("z_img:", z_img.shape)   # Expected: [batch_size, 128]


z_mol: torch.Size([2, 128])
z_img: torch.Size([2, 128])


In [41]:
# ------------------------------------------------------------
# Build a batch of 2 phenotype images
# ------------------------------------------------------------
# For Milestone 13, we use the same image twice to create a
# minimal batch (batch size ≥ 2 is required for contrastive loss).
# ------------------------------------------------------------

img_path = "C:/Users/anjal/OneDrive/Documents/phenotypic-virtual-screening/dual_library_VS.png"

# Load the phenotype image and convert to RGB
img = Image.open(img_path).convert("RGB")

# Apply ImageNet preprocessing (resize, normalize, tensor)
img = transform(img)

# ------------------------------------------------------------
# Encode the image using the ImageEncoder
# ------------------------------------------------------------
# img_encoder(img) returns a tensor of shape [1, 256]
# We remove the extra batch dimension using squeeze(0)
# so the embedding becomes shape [256]
# ------------------------------------------------------------
img_emb = img_encoder(img).squeeze(0)

# Stack two identical embeddings to form a batch of size 2
# Final shape: [2, 256]
img_batch = torch.stack([img_emb, img_emb])

# ------------------------------------------------------------
# Forward pass through the multimodal model
# ------------------------------------------------------------
# mol_batch : batch of molecule embeddings (shape [2, 256])
# img_batch : batch of image embeddings (shape [2, 256])
#
# The model projects both into a shared 128‑dimensional space
# and returns:
#   z_mol : molecule embeddings in shared space
#   z_img : image embeddings in shared space
# ------------------------------------------------------------
z_mol, z_img = model(mol_batch, img_batch)

# Print shapes to verify correct alignment
print("z_mol:", z_mol.shape)   # Expected: [2, 128]
print("z_img:", z_img.shape)   # Expected: [2, 128]

# ------------------------------------------------------------
# Compute CLIP-style contrastive loss
# ------------------------------------------------------------
# This measures how well molecule embeddings align with
# image embeddings in the shared latent space.
# ------------------------------------------------------------
loss = model.contrastive_loss(z_mol, z_img)

# Print the batch loss value
print("Batch contrastive loss:", loss.item())


z_mol: torch.Size([2, 128])
z_img: torch.Size([2, 128])
Batch contrastive loss: 0.6937757134437561


In [42]:
# ------------------------------------------------------------
# Similarity Scoring and Hit Ranking
# ------------------------------------------------------------
# This block computes:
#   1. Cosine similarity between molecule and image embeddings
#   2. Dot‑product similarity (CLIP-style)
#   3. Combined scores (optional)
#   4. Hit ranking for virtual screening
#
# Assumes:
#   - z_mol : [N, 128] molecule embeddings
#   - z_img : [1, 128] or [N, 128] image embeddings
# ------------------------------------------------------------

import torch

# ------------------------------------------------------------
# 1. Cosine similarity
# ------------------------------------------------------------
# Cosine similarity measures directional alignment between
# molecule and image embeddings. Values range from -1 to +1.
# Higher values indicate stronger multimodal alignment.
# ------------------------------------------------------------
cos_sim = F.cosine_similarity(z_mol, z_img)
print("Cosine similarity scores:", cos_sim.tolist())


# ------------------------------------------------------------
# 2. Dot‑product similarity (CLIP-style)
# ------------------------------------------------------------
# Dot‑product similarity is sensitive to both direction and
# magnitude. This is the same similarity used inside CLIP.
# ------------------------------------------------------------
dot_sim = (z_mol * z_img).sum(dim=-1)
print("Dot‑product similarity scores:", dot_sim.tolist())


# ------------------------------------------------------------
# 3. Optional: Combined similarity score
# ------------------------------------------------------------
# You can combine cosine + dot‑product to create a more
# stable ranking metric. This is useful for virtual screening.
# ------------------------------------------------------------
combined_score = 0.5 * cos_sim + 0.5 * dot_sim
print("Combined scores:", combined_score.tolist())


# ------------------------------------------------------------
# 4. Hit Ranking
# ------------------------------------------------------------
# Sort molecules by similarity score (descending).
# This produces a ranked list of molecules most aligned
# with the phenotype image.
# ------------------------------------------------------------
scores = combined_score.detach().cpu()
sorted_indices = torch.argsort(scores, descending=True)

print("\nHit Ranking (best → worst):")
for rank, idx in enumerate(sorted_indices.tolist(), start=1):
    print(f"Rank {rank}: Molecule {idx} | Score = {scores[idx].item():.4f}")


Cosine similarity scores: [0.04113414138555527, 0.03411300852894783]
Dot‑product similarity scores: [0.04113413393497467, 0.03411301225423813]
Combined scores: [0.04113413766026497, 0.03411301225423813]

Hit Ranking (best → worst):
Rank 1: Molecule 0 | Score = 0.0411
Rank 2: Molecule 1 | Score = 0.0341
